# Lab 10A: One Loop, Many Techniques — Looking Inside a Frozen Network

**Before you start: Runtime → Change runtime type → GPU.**

This week's lab consists of **three notebooks**:

| Notebook | Question | Time |
|---|---|---|
| **10A** One loop, many techniques | What can a frozen network tell us when we optimise the *input* instead of the weights? | 50 min |
| **10B** Right for the right reasons? | Our model is 100% accurate on the validation set. Is it looking at the right thing, and can we trust the tools that tell us? | 70 min |
| **10C** *(optional)* Gram matrices, texture and style | What does "style" mean to a network? | 45 min |

**10B is the most important notebook.** If you are short of time, skip 10C, not 10B. Afterwards,
complete this week's pages in your iteration log: you will use the tools from 10B on **your own model**.

**Theory ↔ code.** As in Lab 9, boxes marked **🔗 Theory ↔ code** show how the equations and
procedures from the lecture map to lines of PyTorch.

**Cells marked 🛠️ TASK contain gaps (`???`) that you must fill in.** Most come with a test cell; don't
move on until the tests pass. Write down your ✍️ **Prediction** before running an experiment.

### This notebook (10A)

The lecture presented many techniques: saliency maps, inverting a layer, filter visualisation,
class visualisation, adversarial examples, style transfer. They look very different, but almost all
of them do the **same thing**:

> Freeze the network. Define a scalar loss on some activation or output. Differentiate the loss with
> respect to the **input pixels**, and change the pixels.

In this notebook you write that loop **once**, and then get each technique by changing only the loss.

| Part | You will… |
|---|---|
| 1. Looking inside with hooks | capture the activations of every layer of VGG16 for an image, and measure how they change with depth |
| 2. One loop | write the "optimise the input" loop, and use it for gradient saliency, inversion, filter visualisation, class visualisation and adversarial examples |
| 3. Exam-style questions | check your understanding |

## Setup

In [ ]:
import copy
import math
import random
import time

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import torchvision.transforms.functional as TF

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ---- Images: the example images from the lecture, stored in the course repository ----
import os, urllib.request
from PIL import Image

REPO = 'https://raw.githubusercontent.com/klaverhenrik/Deep-Learning-for-Visual-Recognition-2026/main/data/'
os.makedirs('images', exist_ok=True)


def get_image(name):
    path = os.path.join('images', name)
    if not os.path.exists(path):
        urllib.request.urlretrieve(REPO + name, path)
    return Image.open(path).convert('RGB')


# ---- Preprocessing: the ImageNet normalisation the pretrained networks were trained with ----
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
LOW, HIGH = (0 - IMAGENET_MEAN) / IMAGENET_STD, (1 - IMAGENET_MEAN) / IMAGENET_STD   # pixel 0 and 1 after normalisation


def load_image(name, size=224):
    tfm = T.Compose([T.Resize(size), T.CenterCrop(size), T.ToTensor()])
    x = tfm(get_image(name)).unsqueeze(0).to(device)
    return (x - IMAGENET_MEAN) / IMAGENET_STD             # (1, 3, size, size), normalised


def to_pixels(x):
    # Undo the normalisation, for display: (1, 3, H, W) -> (H, W, 3) in [0, 1]
    return (x * IMAGENET_STD + IMAGENET_MEAN).clamp(0, 1)[0].permute(1, 2, 0).detach().cpu()


def stretch(x):
    # For optimised images that are far outside the natural range: stretch to [0, 1] for display
    x = x[0].detach().cpu()
    return ((x - x.min()) / (x.max() - x.min() + 1e-8)).permute(1, 2, 0)


# ---- VGG16, pretrained on ImageNet (the download is about 0.5 GB, so it takes a little while) ----
weights = torchvision.models.VGG16_Weights.IMAGENET1K_V1
categories = weights.meta['categories']
vgg = torchvision.models.vgg16(weights=weights).to(device).eval()
for p in vgg.parameters():
    p.requires_grad_(False)                     # FREEZE the network: we never change its weights in this lab
for m in vgg.modules():
    if isinstance(m, nn.ReLU):
        m.inplace = False                       # in-place ReLUs would overwrite activations we store

cat = load_image('Cat03.jpg')
with torch.no_grad():
    probs = F.softmax(vgg(cat), dim=1)[0]
top = probs.topk(3)
print('VGG16 says:', [(categories[i], round(p.item(), 2)) for p, i in zip(top.values, top.indices)])
plt.figure(figsize=(3, 3)); plt.imshow(to_pixels(cat)); plt.axis('off'); plt.show()

---
# Part 1 — Looking inside with hooks

To see what happens *inside* a network, we need the outputs of its intermediate layers. PyTorch
lets us attach a **forward hook** to any module: a function that is called with the module's output
every time the module runs.

### 🔗 Theory ↔ code

| Lecture | PyTorch |
|---|---|
| "plot the feature maps of layer $l$" | store the layer's output in a hook, then plot `activation[0, k]` for channel $k$ |
| VGG16's layers `conv1_1`, `relu1_1`, …, `relu5_3` | `vgg.features[0]`, `vgg.features[1]`, …, `vgg.features[29]` (print `vgg.features` to see them all) |
| the first-layer filters, shown as colour images | `vgg.features[0].weight`, shape (64, 3, 3, 3): 64 filters of size 3×3 with 3 colour channels |

### 🛠️ TASK 1a: Hook every ReLU

Register a forward hook (made by `make_hook(i)`) on **every ReLU** in `vgg.features`, and keep the
handles so that we can remove the hooks afterwards.

In [ ]:
activations = {}


def make_hook(i):
    def hook(module, inputs, output):
        activations[i] = output.detach()
    return hook


handles = []
for i, layer in enumerate(vgg.features):
    if ???:                                        # only the ReLU layers
        handles.append(???)                        # register make_hook(i) as a forward hook on this layer

with torch.no_grad():
    vgg(cat)                                       # one forward pass fills `activations`
print('Captured layers:', sorted(activations))

In [ ]:
# Test for Task 1a
assert sorted(activations) == [1, 3, 6, 8, 11, 13, 15, 18, 20, 22, 25, 27, 29], 'you should capture the 13 ReLU layers'
assert activations[29].shape == (1, 512, 14, 14)
print('✅ Task 1a passed')

### How do the activations change with depth?

For each ReLU layer we measure the spatial size, the fraction of activations that are exactly zero,
and the fraction of feature maps that are **completely** zero (all-black) for this image.

✍️ **Prediction:** how will the fraction of zeros change from the first to the last layer?

In [ ]:
names = {1: 'relu1_1', 3: 'relu1_2', 6: 'relu2_1', 8: 'relu2_2', 11: 'relu3_1', 13: 'relu3_2', 15: 'relu3_3',
         18: 'relu4_1', 20: 'relu4_2', 22: 'relu4_3', 25: 'relu5_1', 27: 'relu5_2', 29: 'relu5_3'}
print(f'{"layer":10s}{"shape":>18s}{"zeros":>9s}{"all-zero maps":>15s}')
for i, a in activations.items():
    zeros = (a == 0).float().mean().item()
    dead_maps = (a.flatten(2).amax(2) == 0).float().mean().item()
    print(f'{names[i]:10s}{str(tuple(a.shape[1:])):>18s}{zeros:9.0%}{dead_maps:15.0%}')

fig, axs = plt.subplots(3, 8, figsize=(16, 6.5))
w = vgg.features[0].weight.detach().cpu()
a1, a5 = activations[1][0].cpu(), activations[29][0].cpu()
top5 = a5.flatten(1).amax(1).argsort(descending=True)[:8]
for k in range(8):
    f = w[k]; axs[0, k].imshow(((f - f.min()) / (f.max() - f.min())).permute(1, 2, 0), interpolation='nearest')
    axs[1, k].imshow(a1[k], cmap='gray')
    axs[2, k].imshow(a5[top5[k]], cmap='gray')
for ax in axs.flat:
    ax.axis('off')
axs[0, 0].set_title('conv1_1 filters (3×3, RGB)', loc='left')
axs[1, 0].set_title('relu1_1 feature maps', loc='left')
axs[2, 0].set_title('relu5_3: the 8 most active of 512 feature maps', loc='left')
plt.tight_layout(); plt.show()

### 🛠️ TASK 1b: Why hook the ReLU and not the convolution?

Exercise 2 in the lecture notes hooks the **convolution** layers `features[0]` (conv1_1) and
`features[28]` (conv5_3) and counts exact zeros as "dead". Measure the fraction of exact zeros at the
output of `vgg.features[28]` (hint: register a hook as above, run a forward pass, remove the hook).

In [ ]:
store = {}
h = ???                                            # hook on vgg.features[28] that stores its output in store['conv5_3']
with torch.no_grad():
    vgg(cat)
h.remove()
print(f"conv5_3 (before the ReLU): {(store['conv5_3'] == 0).float().mean().item():.1%} zeros")
print(f"relu5_3 (after the ReLU):  {(activations[29] == 0).float().mean().item():.1%} zeros")

In [ ]:
for h in handles:
    h.remove()                                     # remove the hooks: we don't need them any more

### Questions 1

1. Describe how the fraction of zeros and the fraction of all-zero feature maps change from `relu1_1`
   to `relu5_3`. Compare with the three rows of images.
2. In `relu5_3`, many feature maps are completely black for this cat. Is something wrong with the
   network? What would make you worried?
3. Why does it make no sense to count exact zeros at the output of a convolution layer?
4. What do the first-layer filters look like? Why can we show the first-layer filters as colour images,
   but not the filters of deeper layers?

---
# Part 2 — One loop

### 🔗 Theory ↔ code: training turned around

| | Training (Labs 6–9) | This lab |
|---|---|---|
| What is optimised | the weights $\mathbf{W}$ | the **input image** $\mathbf{x}$ |
| What is fixed | the training images | the network: `requires_grad_(False)` on every weight |
| The optimiser | `Adam(model.parameters())` | `Adam([x])` |
| The loss | how wrong the prediction is | **the question we ask** (see the table below) |
| Extra step | — | a **projection** after each step that keeps $\mathbf{x}$ "valid" (e.g. pixels in $[0, 1]$) |

Because the weights are frozen, `loss.backward()` only computes $\partial \text{loss} / \partial \mathbf{x}$, the
gradient with respect to every pixel. Everything else is the training loop you know.

### 🛠️ TASK 2a: Write the loop

`loss_fn(x)` returns a scalar; `project(x, step)` (optional) modifies `x` in place after each step.
Both are given by the caller, so the same loop works for every technique below.

In [ ]:
def optimise_input(loss_fn, x_init, steps, lr, project=None):
    x = x_init.clone().requires_grad_(True)        # the image is now the "parameter"
    opt = torch.optim.Adam([x], lr=lr)
    for step in range(steps):
        ???                                        # 1. reset the gradient
        loss = ???                                 # 2. compute the loss for the current image
        ???                                        # 3. backpropagate: d loss / d x
        ???                                        # 4. update the pixels
        if project is not None:
            with torch.no_grad():
                project(x, step)                   # 5. keep the image valid
    return x.detach()

In [ ]:
# Tests for Task 2a
_x = optimise_input(lambda x: ((x - 3) ** 2).mean(), torch.zeros(1, 3, 4, 4), steps=500, lr=0.1)
assert torch.allclose(_x, torch.full_like(_x, 3.0), atol=0.05), 'without projection, x should go to the minimum (3)'
_x = optimise_input(lambda x: ((x - 3) ** 2).mean(), torch.zeros(1, 3, 4, 4), steps=500, lr=0.1,
                    project=lambda x, step: x.clamp_(0, 1))
assert torch.allclose(_x, torch.ones_like(_x), atol=1e-6), 'with the projection, x should end at the boundary (1)'
_x = optimise_input(lambda x: -vgg(x)[0, 0], cat, steps=1, lr=0.01)
assert all(p.grad is None for p in vgg.parameters()), "the network's weights must not get gradients"
print('✅ Task 2a passed')


def keep_valid(x, step):
    # The projection we use most: every pixel stays between 0 and 1 (in normalised units: LOW and HIGH)
    x.copy_(torch.max(torch.min(x, HIGH), LOW))

### 2.1 Gradient saliency: the loop with zero steps

The simplest technique doesn't change the image at all: we compute the gradient of the class score
$y_c$ (the **logit**, before the softmax) with respect to the pixels once, and show its size:
$\text{saliency}_{ij} = \max_\text{colour} \left| \partial y_c / \partial x_{ij} \right|$.

### 🛠️ TASK 2b: Gradient saliency

In [ ]:
def gradient_saliency(model, x, class_idx):
    x = x.clone().requires_grad_(True)
    score = ???                                    # the logit of class_idx for this (single) image
    score.backward()
    return ???                                     # |gradient|, maximum over the colour channels: shape (H, W)


cat_idx = vgg(cat).argmax().item()
sal = gradient_saliency(vgg, cat, cat_idx)
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
axs[0].imshow(to_pixels(cat)); axs[1].imshow(sal.cpu(), cmap='gray', vmax=sal.quantile(0.99).item())
axs[1].set_title(f'gradient saliency: {categories[cat_idx]}')
for ax in axs: ax.axis('off')
plt.show()

In [ ]:
# Test for Task 2b
assert sal.shape == (224, 224) and (sal >= 0).all()
print('✅ Task 2b passed')

### 2.2 Inverting a layer: what does a layer remember?

Given the activations $\mathbf{a}^l(\mathbf{x})$ of the cat at layer $l$, find an image $\mathbf{y}$ with the same
activations, starting from noise (lecture, Section 5.1):

$$\mathcal{L}(\mathbf{y}) = \frac{\|\mathbf{a}^l(\mathbf{y}) - \mathbf{a}^l(\mathbf{x})\|^2}{\|\mathbf{a}^l(\mathbf{x})\|^2} \; + \; \lambda_\text{TV} \cdot \text{TV}(\mathbf{y}).$$

We divide by $\|\mathbf{a}^l(\mathbf{x})\|^2$ so that the same learning rate works for every layer. The
second term is optional: the **total variation** $\text{TV}(\mathbf{y})$ is the average absolute difference
between neighbouring pixels, which penalises high-frequency noise. It is a regulariser for the
*image*, just as weight decay is a regulariser for the weights: many images have the same
activations, and the TV term prefers the smooth ones.

### 🛠️ TASK 2c: The inversion loss

`front` is the network up to and including layer $l$. Fill in the loss.

✍️ **Prediction:** what will the reconstructions from `relu1_2` and from `relu5_3` look like?

In [ ]:
def total_variation(x):
    return (x[..., 1:, :] - x[..., :-1, :]).abs().mean() + (x[..., :, 1:] - x[..., :, :-1]).abs().mean()


def invert(layer_idx, tv_weight, steps=200):
    front = vgg.features[:layer_idx + 1]
    with torch.no_grad():
        target = front(cat)
    loss_fn = lambda y: ???                        # normalised squared distance between front(y) and target, + tv_weight * TV(y)
    return optimise_input(loss_fn, 0.01 * torch.randn_like(cat), steps=steps, lr=0.05, project=keep_valid)


layers = {3: 'relu1_2', 8: 'relu2_2', 15: 'relu3_3', 22: 'relu4_3', 29: 'relu5_3'}
t0 = time.time()
fig, axs = plt.subplots(2, 6, figsize=(18, 6.5))
for r, tv in enumerate([0.0, 0.1]):
    axs[r, 0].imshow(to_pixels(cat)); axs[r, 0].set_title('original')
    for j, (idx, name) in enumerate(layers.items()):
        axs[r, j + 1].imshow(to_pixels(invert(idx, tv))); axs[r, j + 1].set_title(f'from {name}, TV weight {tv}')
for ax in axs.flat: ax.axis('off')
plt.tight_layout(); plt.show()
print(f'Time: {time.time() - t0:.0f} s')

### Questions 2.2

1. Describe how the reconstructions change from `relu1_2` to `relu5_3`. What is kept, and what is lost?
2. Every reconstruction has (almost) the same activations as the original at that layer. What does that
   tell you about the network: is it invertible?
3. What did the TV term change? Is the TV-regularised image a "more correct" reconstruction?

### 2.3 Filter visualisation: what does one filter want to see?

Start from noise and change the image so that the **mean activation of one filter** becomes as large
as possible. The optimiser minimises, so the loss is the *negative* mean activation (gradient
**ascent** on the activation). After each step we keep the pixels valid, and every 10 steps we blur
the image slightly, to suppress high-frequency noise (the same role as the TV term above).

### 🛠️ TASK 2d: The filter-visualisation loss

In [ ]:
def visualise_filter(layer_idx, filter_idx, steps=150):
    front = vgg.features[:layer_idx + 1]
    loss_fn = lambda x: ???                        # NEGATIVE mean activation of channel filter_idx of front(x)

    def project(x, step):
        keep_valid(x, step)
        if step % 10 == 9:
            x.copy_(TF.gaussian_blur(x, kernel_size=3, sigma=0.5))

    return optimise_input(loss_fn, 0.01 * torch.randn_like(cat), steps=steps, lr=0.1, project=project)


t0 = time.time()
fig, axs = plt.subplots(3, 4, figsize=(12, 9.5))
for r, (idx, name) in enumerate({0: 'conv1_1', 14: 'conv3_3', 28: 'conv5_3'}.items()):
    for c, f in enumerate([0, 1, 2, 3]):
        axs[r, c].imshow(stretch(visualise_filter(idx, f))); axs[r, c].set_title(f'{name}, filter {f}'); axs[r, c].axis('off')
plt.tight_layout(); plt.show()
print(f'Time: {time.time() - t0:.0f} s')

### 2.4 Class visualisation: what does the network think a "dumbbell" looks like?

The same idea with the **class score** as the objective (class-based reconstruction): find an image
that maximises the logit $y_c$ of one class. Without help, this produces high-frequency noise that the
network happens to like, so we add three regularisers:

* a small penalty on the size of the image, $\lambda \|\mathbf{x}\|^2$ (like weight decay, but for pixels);
* a slight blur every 10 steps (as for the filters);
* **jitter**: at every step, the image is shifted by a few random pixels before it goes into the network.
  A pattern only survives if it raises the score *wherever* it is, which suppresses pixel-exact noise.

The lecture showed that for "dumbbell", such images can contain arms. Let's see for ourselves.

### 🛠️ TASK 2e: The class-visualisation loss

In [ ]:
def visualise_class(class_idx, steps=200, l2=1e-4):
    def loss_fn(x):
        dx, dy = random.randint(-8, 8), random.randint(-8, 8)
        x_shifted = torch.roll(x, shifts=(dx, dy), dims=(2, 3))  # jitter
        return ???                                 # NEGATIVE logit of class_idx for x_shifted, plus l2 * (sum of squared pixels of x)

    def project(x, step):
        keep_valid(x, step)
        if step % 10 == 9:
            x.copy_(TF.gaussian_blur(x, kernel_size=3, sigma=0.5))

    return optimise_input(loss_fn, 0.01 * torch.randn_like(cat), steps=steps, lr=0.1, project=project)


t0 = time.time()
classes = [543, 1, 949]                                  # dumbbell, goldfish, strawberry
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
for ax, c in zip(axs, classes):
    x = visualise_class(c)
    with torch.no_grad():
        p = F.softmax(vgg(x), dim=1)[0, c].item()
    ax.imshow(to_pixels(x)); ax.set_title(f'{categories[c]}  (VGG16: p = {p:.2f})'); ax.axis('off')
plt.tight_layout(); plt.show()
print(f'Time: {time.time() - t0:.0f} s')

### Questions 2.3–2.4

1. Every `conv1_1` image is a **single uniform colour**. Why? (Hint: `conv1_1` is one linear 3×3 filter, and
   the loss is its *mean* output over all positions. What does that mean depend on?) Is this what the
   `conv1_1` filter "detects"?
2. How do the preferred patterns change from `conv3_3` to `conv5_3`?
3. Can you recognise the classes in the class visualisations? Do you see parts that are not the object
   itself (like the arms for "dumbbell")? What would that tell you about the training data?
4. VGG16 is (almost) certain that these images show the class. Would a human agree? What does that tell
   you about the difference between "an image the network is confident about" and "a typical image of
   the class"?

### 2.5 Adversarial examples: the same loop, used against the network

An **adversarial example** is an image that looks unchanged to a human but changes the network's
prediction. The Fast Gradient Sign Method (FGSM) takes a **single** step: compute the gradient of the
loss for the correct (here: the predicted) class, and move every pixel by $\pm\varepsilon$ in the direction
that *increases* the loss:

$$\mathbf{x}_\text{adv} = \mathbf{x} + \varepsilon \cdot \text{sign}\left(\nabla_\mathbf{x} \mathcal{L}(\mathbf{x}, y)\right).$$

$\varepsilon$ is measured in **pixel units** (pixels lie in $[0, 1]$, so $\varepsilon = 0.01$ means "change each pixel by
at most 1% of its range", invisible to a human). Our images are normalised, so in code a pixel change of
$\varepsilon$ is a change of $\varepsilon / \text{std}$ in the normalised values.

We measure **how often the prediction changes** for a set of images, as a function of $\varepsilon$, and whether
the same adversarial images also fool a **different** network (ResNet-18), which was never used to
compute them (transfer).

### 🛠️ TASK 2f: FGSM

In [ ]:
names = ['Cat03.jpg', 'Cute_dog.jpg', 'Collage_of_Nine_Dogs.jpg', 'LeafySeaDragon.jpg', 'Good_Food_Display_-_NCI_Visuals_Online.jpg',
         'Kitano_Street_Kobe01s5s4110.jpg', 'Inside_the_shopping_mall_CC_Mart.jpg', 'paris.jpg', 'cat_and_dog.jpg', 'starry_night.jpg']
X = torch.cat([load_image(n) for n in names])
resnet = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1).to(device).eval()
for p in resnet.parameters():
    p.requires_grad_(False)
with torch.no_grad():
    pred_vgg, pred_resnet = vgg(X).argmax(1), resnet(X).argmax(1)


def fgsm(model, x, y, eps):
    x = x.clone().requires_grad_(True)
    loss = F.cross_entropy(model(x), y)
    loss.backward()
    x_adv = ???                                    # one step of size eps (in PIXEL units) along the sign of the gradient
    return torch.max(torch.min(x_adv, HIGH), LOW).detach()      # keep the pixels valid


print(f'{"eps":>6s} {"VGG16 changed":>14s} {"ResNet-18 changed (transfer)":>30s}')
for eps in [0.0, 0.001, 0.003, 0.01, 0.03]:
    x_adv = fgsm(vgg, X, pred_vgg, eps)
    with torch.no_grad():
        changed_vgg = (vgg(x_adv).argmax(1) != pred_vgg).float().mean().item()
        changed_resnet = (resnet(x_adv).argmax(1) != pred_resnet).float().mean().item()
    print(f'{eps:6.3f} {changed_vgg:14.0%} {changed_resnet:30.0%}')

### A targeted attack with the same loop

FGSM only asks for *any* wrong prediction. With `optimise_input` we can ask for a **specific** one: make
VGG16 call the cat a "goldfish", while no pixel changes by more than $\varepsilon = 0.01$. The constraint is
just another projection.

✍️ **Prediction:** will you be able to see the difference between the two images?

In [ ]:
eps = 0.01
goldfish = 1


def stay_close(x, step):
    delta = torch.max(torch.min(x - cat, eps / IMAGENET_STD), -eps / IMAGENET_STD)    # |change| <= eps per pixel
    x.copy_(torch.max(torch.min(cat + delta, HIGH), LOW))


x_target = optimise_input(lambda x: -vgg(x)[0, goldfish], cat, steps=50, lr=0.01, project=stay_close)
with torch.no_grad():
    p = F.softmax(vgg(x_target), dim=1)[0]
diff = (to_pixels(x_target) - to_pixels(cat)).abs()
fig, axs = plt.subplots(1, 3, figsize=(13, 4.5))
axs[0].imshow(to_pixels(cat)); axs[0].set_title(f'original: {categories[cat_idx]} ({probs[cat_idx]:.2f})')
axs[1].imshow(to_pixels(x_target)); axs[1].set_title(f'attacked: {categories[p.argmax()]} ({p.max():.2f})')
axs[2].imshow(diff / diff.max()); axs[2].set_title(f'difference, amplified (max {diff.max():.3f})')
for ax in axs: ax.axis('off')
plt.show()

### Questions 2.5

1. How large must $\varepsilon$ be to change most of VGG16's predictions? Would you see a change of that size
   in an image?
2. Did the adversarial images also fool ResNet-18? What does it mean that adversarial examples
   **transfer** between networks?
3. Adversarial attacks and class visualisation use the same loop. What is the one difference that turns
   a visualisation into an attack?
4. **Project perspective:** why should anyone who deploys a vision model care about this, even if nobody
   is trying to attack it?

### 2.6 Summary: one loop, many questions

Fill in the table from your own code. The lecture's summary table (Section 10 of the notes) is the
reference.

| Technique | Loss (`loss_fn`) | Steps | Projection | What it reveals |
|---|---|---|---|---|
| Gradient saliency | | | | |
| Layer inversion | | | | |
| Filter visualisation | | | | |
| Class visualisation | | | | |
| FGSM | | | | |
| Targeted attack | | | | |

---
# Part 3 — Exam-style questions

**Q1.** In all the techniques of this notebook, what is optimised and what is kept fixed?

* (a) The weights are optimised; the input image is fixed.
* (b) The input image is optimised; the weights are fixed.
* (c) Both the weights and the image are optimised.
* (d) Nothing is optimised; we only run forward passes.

**Q2.** Why is the loss in filter visualisation the **negative** mean activation of the filter?

* (a) Because activations after a ReLU are negative.
* (b) Because the optimiser minimises the loss, and we want to *maximise* the activation.
* (c) To make the image darker.
* (d) Because the gradient of the mean is negative.

**Q3.** A layer inversion from `relu5_3` gives an image that looks quite different from the original, yet
has almost the same `relu5_3` activations. What does this show?

* (a) The optimisation failed.
* (b) The network is invertible.
* (c) Many different images map to the same deep activations: deep layers discard information such as exact colours and details.
* (d) `relu5_3` is a dead layer.

**Q4.** In FGSM, $\mathbf{x}_\text{adv} = \mathbf{x} + \varepsilon\cdot\text{sign}(\nabla_\mathbf{x}\mathcal{L})$. What does $\varepsilon$ control?

* (a) The learning rate of the network.
* (b) The maximum change of each pixel.
* (c) The number of attack steps.
* (d) Which class the attack targets.

---
**Next: notebook 10B**, where we use these tools to find out whether a model is right for the right
reasons.